In [3]:
import pandas as pd

clean = pd.read_csv("teaching_prices_cleaned.csv", parse_dates=["date"])
clean["flag"] = clean["flag"].fillna("")
print(clean.shape)


(96, 7)


In [4]:
# Checks: 96 rows, each item and city 12 times, 3 flagged missing prices
print("Rows:", len(clean))
print("Missing price:", clean["price"].isna().sum())
print(clean.groupby(["item", "city"]).size())

Rows: 96
Missing price: 3
item         city  
Oil B        City 1    12
             City 2    12
Pulse C      City 1    12
             City 2    12
Staple A     City 1    12
             City 2    12
Sweetener D  City 1    12
             City 2    12
dtype: int64


In [5]:
stats = clean.groupby("item")["price"].agg(["count", "mean", "median", "std", "min", "max"]).round(2)
stats["mean_minus_median"] = (stats["mean"] - stats["median"]).round(2)
stats

,count,mean,median,std,min,max,mean_minus_median
item,,,,,,,
Oil B,24,324.83,324.0,14.53,298.0,350.0,0.83
Pulse C,23,205.91,206.0,3.91,200.0,213.5,-0.09
Staple A,23,111.26,112.5,6.27,99.5,122.0,-1.24
Sweetener D,23,167.15,167.5,9.70,149.0,184.5,-0.35


In [6]:
# First-to-last change, by item and city (1 Jan to 19 Mar)
first_date = clean["date"].min()
last_date = clean["date"].max()
print("First date:", first_date.date(), " Last date:", last_date.date())

f = clean[clean["date"] == first_date].set_index(["item", "city"])["price"]
l = clean[clean["date"] == last_date].set_index(["item", "city"])["price"]
change = pd.DataFrame({"first": f, "last": l})
change["change_pkr"] = change["last"] - change["first"]
change["pct_change"] = (change["change_pkr"] / change["first"] * 100).round(2)
change

First date: 2026-01-01  Last date: 2026-03-19


first   last  change_pkr  pct_change
item        city                                        
Staple A    City 1  101.0  114.5        13.5       13.37
            City 2  107.0  120.5        13.5       12.62
Oil B       City 1  298.0  344.0        46.0       15.44
            City 2  304.0  350.0        46.0       15.13
Pulse C     City 1  200.0  207.5         7.5        3.75
            City 2    NaN  213.5         NaN         NaN
Sweetener D City 1  149.0  178.5        29.5       19.80
            City 2  155.0  184.5        29.5       19.03

In [7]:
# Is City 2 always 6 more than City 1? (to check if City 1 alone is fair)
wide = clean.pivot_table(index=["date", "item"], columns="city", values="price")
gap = wide["City 2"] - wide["City 1"]
print(gap.describe())

count    45.0
mean      6.0
std       0.0
min       6.0
25%       6.0
50%       6.0
75%       6.0
max       6.0
dtype: float64


In [8]:
# Summary table I will paste to the AI in Part 2
table = stats[["count", "mean", "median", "min", "max"]].copy()
c1 = change.xs("City 1", level="city")
table["first_City1"] = c1["first"]
table["last_City1"] = c1["last"]
table["pct_change_City1"] = c1["pct_change"]
table

,count,mean,median,min,max,first_City1,last_City1,pct_change_City1
item,,,,,,,,
Oil B,24,324.83,324.0,298.0,350.0,298.0,344.0,15.44
Pulse C,23,205.91,206.0,200.0,213.5,200.0,207.5,3.75
Staple A,23,111.26,112.5,99.5,122.0,101.0,114.5,13.37
Sweetener D,23,167.15,167.5,149.0,184.5,149.0,178.5,19.80


In [9]:
staple = clean[clean["item"] == "Staple A"].pivot_table(index="date", columns="city", values="price")
staple["week_change_City1"] = staple["City 1"].diff()
staple

city,City 1,City 2,week_change_City1
date,,,
2026-01-01,101.0,107.0,NaN
2026-01-08,99.5,105.5,-1.5
2026-01-15,103.0,NaN,3.5
2026-01-22,106.5,112.5,3.5
2026-01-29,105.0,111.0,-1.5
2026-02-05,108.5,114.5,3.5
2026-02-12,107.0,113.0,-1.5
2026-02-19,110.5,116.5,3.5
2026-02-26,114.0,120.0,3.5


In [10]:
# How many weeks did Staple A City 1 go down?
print("Weeks down:", (staple["week_change_City1"] < 0).sum())
print(staple[staple["week_change_City1"] < 0])

Weeks down: 5
city        City 1  City 2  week_change_City1
date                                         
2026-01-08    99.5   105.5               -1.5
2026-01-29   105.0   111.0               -1.5
2026-02-12   107.0   113.0               -1.5
2026-03-05   112.5   118.5               -1.5
2026-03-19   114.5   120.5               -1.5


In [11]:
# Mean and median of the first 4 weeks vs the last 4 weeks, for each item
dates = sorted(clean["date"].unique())
early = clean[clean["date"].isin(dates[:4])].groupby("item")["price"].agg(["mean", "median"])
late = clean[clean["date"].isin(dates[-4:])].groupby("item")["price"].agg(["mean", "median"])
fl = pd.DataFrame({
    "early_mean": early["mean"], "late_mean": late["mean"],
    "early_median": early["median"], "late_median": late["median"],
})
fl["pct_change_mean"] = ((fl["late_mean"] - fl["early_mean"]) / fl["early_mean"] * 100).round(2)
fl["pct_change_median"] = ((fl["late_median"] - fl["early_median"]) / fl["early_median"] * 100).round(2)
fl.round(2)

,early_mean,late_mean,early_median,late_median,pct_change_mean,pct_change_median
item,,,,,,
Oil B,308.75,340.50,310.0,340.50,10.28,9.84
Pulse C,204.00,208.00,202.5,208.00,1.96,2.72
Staple A,105.00,117.25,105.5,117.25,11.67,11.14
Sweetener D,156.07,177.25,155.0,177.25,13.57,14.35


In [12]:
# The 12-week mean and median of Staple A, compared to its first and last price
print("Staple A 12-week mean:", round(clean[clean["item"] == "Staple A"]["price"].mean(), 2))
print("Staple A 12-week median:", clean[clean["item"] == "Staple A"]["price"].median())
print("Staple A City 1 first and last:", staple["City 1"].iloc[0], staple["City 1"].iloc[-1])

Staple A 12-week mean: 111.26
Staple A 12-week median: 112.5
Staple A City 1 first and last: 101.0 114.5


In [13]:
# Does the % change depend on which end date I pick? (Staple A, City 1)
for end in ["2026-03-12", "2026-03-19"]:
    end_price = staple.loc[end, "City 1"]
    print(end, end_price, round((end_price - 101.0) / 101.0 * 100, 2), "%")

2026-03-12 116.0 14.85 %
2026-03-19 114.5 13.37 %


## Part 1c: Mean and median with and without a flagged value
Item: Staple A. Flagged value: "1,250" on 2026-01-15, City 2 (set to missing in Week 2).

In [14]:
sa = clean[clean["item"] == "Staple A"]["price"]
without = sa.dropna()
with_flag = pd.concat([without, pd.Series([1250.0])])

comp = pd.DataFrame({
    "count": [without.count(), with_flag.count()],
    "mean": [round(without.mean(), 2), round(with_flag.mean(), 2)],
    "median": [without.median(), with_flag.median()],
}, index=["without 1,250", "with 1,250"])
comp

,count,mean,median
"without 1,250",23,111.26,112.5
"with 1,250",24,158.71,112.5


In [15]:
# How much did one value move each statistic?
print("Mean moved by:", round(with_flag.mean() - without.mean(), 2))
print("Median moved by:", with_flag.median() - without.median())
print("Highest real Staple A price:", without.max())

Mean moved by: 47.45
Median moved by: 0.0
Highest real Staple A price: 122.0


In [16]:
# Same test on Sweetener D with the 9999 code
sd = clean[clean["item"] == "Sweetener D"]["price"].dropna()
sd_with = pd.concat([sd, pd.Series([9999.0])])
pd.DataFrame({
    "mean": [round(sd.mean(), 2), round(sd_with.mean(), 2)],
    "median": [sd.median(), sd_with.median()],
}, index=["without 9999", "with 9999"])

,mean,median
without 9999,167.15,167.50
with 9999,576.81,167.75


## Part 3: Recompute every number in the AI's interpretation
Each cell matches one sentence (S1 to S12) in the audit in 3_log_reflection.md.

In [17]:
# S1: Oil B most expensive, mean 324.83, median 324.0
print(stats.sort_values("mean", ascending=False)[["mean", "median"]])

               mean  median
item                       
Oil B        324.83   324.0
Pulse C      205.91   206.0
Sweetener D  167.15   167.5
Staple A     111.26   112.5


In [18]:
# S2: mean and median within about 1.5 PKR for every item
print(stats["mean_minus_median"])
print("Largest gap:", stats["mean_minus_median"].abs().max())

item
Oil B          0.83
Pulse C       -0.09
Staple A      -1.24
Sweetener D   -0.35
Name: mean_minus_median, dtype: float64
Largest gap: 1.24


In [19]:
# S3: Sweetener D largest % increase, about 19 to 20%
# S4: Pulse C rose only about 3.75%
# S5: Staple A rose about 13%
print(change.sort_values("pct_change", ascending=False))

                    first   last  change_pkr  pct_change
item        city                                        
Sweetener D City 1  149.0  178.5        29.5       19.80
            City 2  155.0  184.5        29.5       19.03
Oil B       City 1  298.0  344.0        46.0       15.44
            City 2  304.0  350.0        46.0       15.13
Staple A    City 1  101.0  114.5        13.5       13.37
            City 2  107.0  120.5        13.5       12.62
Pulse C     City 1  200.0  207.5         7.5        3.75
            City 2    NaN  213.5         NaN         NaN


In [25]:

g = gap.dropna()  # 3 pairs have a missing price
print("Pairs compared:", len(g))
print("City 2 higher on every date and item:", (g > 0).all(), "| gap values:", g.unique())

Pairs compared: 45
City 2 higher on every date and item: True | gap values: [6.]


In [26]:

print(stats.loc["Staple A", ["mean", "median"]])

mean      111.26
median    112.50
Name: Staple A, dtype: float64


In [27]:

print(staple[["City 1", "City 2", "week_change_City1"]])

city        City 1  City 2  week_change_City1
date                                         
2026-01-01   101.0   107.0                NaN
2026-01-08    99.5   105.5               -1.5
2026-01-15   103.0     NaN                3.5
2026-01-22   106.5   112.5                3.5
2026-01-29   105.0   111.0               -1.5
2026-02-05   108.5   114.5                3.5
2026-02-12   107.0   113.0               -1.5
2026-02-19   110.5   116.5                3.5
2026-02-26   114.0   120.0                3.5
2026-03-05   112.5   118.5               -1.5
2026-03-12   116.0   122.0                3.5
2026-03-19   114.5   120.5               -1.5


In [28]:

print(clean["data_label"].unique())
print(clean["unit"].unique())
print("Cities:", clean["city"].unique(), " Items:", clean["item"].nunique(), " Weeks:", clean["date"].nunique())

['ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS']
['PKR per kg (illustrative)']
Cities: ['City 1' 'City 2']  Items: 4  Weeks: 12


In [29]:

print(clean.columns.tolist())

['date', 'item', 'city', 'price', 'unit', 'data_label', 'flag']
